In [3]:
# Homework 4 - Problem 1

import numpy as np

# -----------------------------
# Parameters (FROM HW3)
# -----------------------------
S0 = 1000
K = 1000
T = 1
N = 60
r = 0.05

dt = T / N

# Given moves (NOT CRR via sigma)
u = 1.005
d = 0.997

# Risk-neutral probability
def risk_neutral_q(r):
    return ((1 + r * dt) - d) / (u - d)

# -----------------------------
# Build stock tree
# -----------------------------
def build_stock_tree():
    S = np.zeros((N+1, N+1))
    for i in range(N+1):
        for j in range(i+1):
            S[i, j] = S0 * (u**j) * (d**(i-j))
    return S

# -----------------------------
# European Put
# -----------------------------
def european_put(S, r):
    q = risk_neutral_q(r)
    V = np.zeros((N+1, N+1))

    # terminal payoff
    for j in range(N+1):
        V[N, j] = max(K - S[N, j], 0)

    # backward induction
    for i in range(N-1, -1, -1):
        for j in range(i+1):
            V[i, j] = (1 / (1 + r * dt)) * (
                q * V[i+1, j+1] + (1 - q) * V[i+1, j]
            )

    return V

# -----------------------------
# American Put
# -----------------------------
def american_put(S, r, tol=1e-10):
    q = risk_neutral_q(r)
    V = np.zeros((N+1, N+1))

    # terminal payoff
    for j in range(N+1):
        V[N, j] = max(K - S[N, j], 0)

    earliest_exercise = None

    # backward induction
    for i in range(N-1, -1, -1):
        for j in range(i+1):

            continuation = (1 / (1 + r * dt)) * (
                q * V[i+1, j+1] + (1 - q) * V[i+1, j]
            )

            exercise = K - S[i, j]

            V[i, j] = max(exercise, continuation)

            if exercise > continuation + tol:
                if earliest_exercise is None or i < earliest_exercise:
                    earliest_exercise = i

    return V, earliest_exercise

# -----------------------------
# RUN PART (a) and (b)
# -----------------------------
S = build_stock_tree()

V_eu = european_put(S, r)
V_am, earliest = american_put(S, r)

print("===== PART (a) =====")
print(f"European Put: {V_eu[0,0]:.6f}")
print(f"American Put: {V_am[0,0]:.6f}")

print("\n===== PART (b) =====")
if earliest is not None:
    print(f"Earliest exercise step: {earliest}")
    print(f"Time: {earliest * dt:.4f} years")
else:
    print("No early exercise")

# -----------------------------
# PART (c): r = 0
# -----------------------------
r0 = 0.0

V_eu_0 = european_put(S, r0)
V_am_0, earliest_0 = american_put(S, r0)

print("\n===== PART (c): r = 0 =====")
print(f"European Put (r=0): {V_eu_0[0,0]:.6f}")
print(f"American Put (r=0): {V_am_0[0,0]:.6f}")

if earliest_0 is not None:
    print(f"Earliest exercise step (r=0): {earliest_0}")
    print("Early exercise may appear due to discrete model effects.")
else:
    print("No early exercise")

# difference check
diff = abs(V_am_0[0,0] - V_eu_0[0,0])
print(f"\nDifference (Am - Eu) at r=0: {diff:.10f}")

===== PART (a) =====
European Put: 0.666959
American Put: 3.293502

===== PART (b) =====
Earliest exercise step: 3
Time: 0.0500 years

===== PART (c): r = 0 =====
European Put (r=0): 12.012689
American Put (r=0): 12.012689
No early exercise

Difference (Am - Eu) at r=0: 0.0000000000
